# Pipeline 05: Interactive UI & Data Flywheel

**Goal:** Gradio web UI for interactive editing + logging verified (prompt, JSON) pairs for distillation.

**Depends on:** Pipelines 01-04, `shared.py`

**Runtime:** GPU required (SAM). Launches a shareable URL.

In [2]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/photo-style-rl'

!pip install gradio anthropic segment-anything opencv-python-headless kornia -q

import shutil
shutil.copy(f'{PROJECT}/src/shared.py', '/content/shared.py')

import os, json, cv2, torch, copy
import numpy as np
import gradio as gr
from PIL import Image
from google.colab import userdata
from anthropic import Anthropic
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

from shared import (
    PROJECT, CHECKPOINTS_DIR, DATA_DIR, RAW_DIR, EDITED_DIR,
    pair_files, extract_json, image_to_base64, resize_for_sam,
    DeterministicRenderer, segment_and_label, feather_mask,
    load_style_profile, merge_params, AVAILABLE_REGIONS
)

DATASET_PATH = f'{DATA_DIR}/slm_training_dataset.jsonl'
os.makedirs(DATA_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

SAM_CHECKPOINT = f'{CHECKPOINTS_DIR}/sam_vit_h_4b8939.pth'
sam = sam_model_registry['vit_h'](checkpoint=SAM_CHECKPOINT).to(device)
auto_generator = SamAutomaticMaskGenerator(
    sam, points_per_side=32, pred_iou_thresh=0.86,
    stability_score_thresh=0.92, min_mask_region_area=0,
)

advanced_profile = load_style_profile('simon_advanced_color_profile.json')
renderer = DeterministicRenderer()

# Use YOUR photos as test images (saved to Drive, not ephemeral Unsplash downloads)
pairs = pair_files()
test_images = {}
for i, (raw_f, _) in enumerate(pairs[:5]):
    name = f"Photo {i+1}: {raw_f}"
    test_images[name] = os.path.join(RAW_DIR, raw_f)

print(f"Ready. {len(test_images)} test images from your dataset.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.4 MB/s eta 0:00:00
Ready. 5 test images from your dataset.


In [3]:
# Backend
def process_edit(image_choice, prompt):
    """Parse prompt, apply rendering, return result + JSON."""
    img_path = test_images.get(image_choice)
    if not img_path or not os.path.exists(img_path):
        return None, "{}", "Image not found."
    if not prompt.strip():
        return Image.open(img_path), "{}", "Enter a prompt."

    try:
        # 1. Claude translates prompt to JSON
        response = client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=500,
            system=f"""Translate photo editing instructions into JSON overrides.
Available regions: {list(advanced_profile.keys())}
Output ONLY JSON: {{"region_or_global": {{"tone_curve": {{}}, "color_mixer": {{}}, exposure: num, ...}}}}""",
            messages=[{"role": "user", "content": prompt}],
        )
        llm_overrides = extract_json(response.content[0].text) or {}
        formatted_json = json.dumps(llm_overrides, indent=2)
    except Exception as e:
        return Image.open(img_path), "{}", f"LLM error: {e}"

    try:
        # 2. Render: base profile + overrides
        img = Image.open(img_path).convert("RGB")
        img_resized = resize_for_sam(img)
        img_np = np.array(img_resized).astype(np.float32) / 255.0

        # Apply base profile globally (use 'subject' or first available)
        base_key = 'subject' if 'subject' in advanced_profile else list(advanced_profile.keys())[0]
        base = copy.deepcopy(advanced_profile.get(base_key, {}))
        for region, override in llm_overrides.items():
            base = merge_params(base, override)

        result_np = renderer.render(img_np, base)
        result = Image.fromarray((np.clip(result_np, 0, 1) * 255).astype(np.uint8))

        return result, formatted_json, "Render complete. Save if accurate."
    except Exception as e:
        return Image.open(img_path), formatted_json, f"Render error: {e}"


def save_to_dataset(prompt, json_output):
    """Append verified (prompt, JSON) pair to training dataset."""
    if not prompt.strip() or not json_output or json_output == "{}":
        return "Cannot save empty data."
    try:
        parsed = json.loads(json_output)
        record = {
            "messages": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": json.dumps(parsed)},
            ]
        }
        with open(DATASET_PATH, "a") as f:
            f.write(json.dumps(record) + "\n")
        count = sum(1 for _ in open(DATASET_PATH))
        return f"Saved. Total records: {count}"
    except Exception as e:
        return f"Error: {e}"


def undo_last_save():
    """Remove the last record from the dataset."""
    if not os.path.exists(DATASET_PATH):
        return "Dataset not found."
    with open(DATASET_PATH, 'r') as f:
        lines = f.readlines()
    if not lines:
        return "Dataset empty."
    lines.pop()
    with open(DATASET_PATH, 'w') as f:
        f.writelines(lines)
    return f"Removed last record. {len(lines)} remaining."

In [4]:
# Gradio UI
with gr.Blocks(theme=gr.themes.Monochrome()) as app:
    gr.Markdown("# Photo Style Editor + Data Flywheel")
    gr.Markdown("Test the pipeline on your photos. Verified results get logged for SLM distillation.")

    with gr.Row():
        with gr.Column(scale=1):
            image_selector = gr.Radio(choices=list(test_images.keys()), value=list(test_images.keys())[0], label="Image")
            prompt_input = gr.Textbox(lines=3, placeholder="e.g., warm up the highlights, desaturate blues...", label="Prompt")
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("### Claude Output")
            json_output = gr.Code(language="json", label="Generated JSON", lines=12)

            with gr.Row():
                save_btn = gr.Button("Save to Dataset", variant="secondary")
                undo_btn = gr.Button("Undo Last Save")
            status = gr.Markdown()

        with gr.Column(scale=1):
            image_output = gr.Image(label="Result", type="pil")

    generate_btn.click(process_edit, [image_selector, prompt_input], [image_output, json_output, status])
    save_btn.click(save_to_dataset, [prompt_input, json_output], [status])
    undo_btn.click(undo_last_save, [], [status])

app.launch(share=True, debug=False)

/tmp/ipykernel_20560/1315608631.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae31bf38719395c747.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
